# 05. Cold protein split: do the arms still tie when the test proteins are unseen?

On the random pair split every protein is in both train and test, and the random-vector control showed that most of the score comes from recognizing the protein. Here whole proteins are held out:

- `cold_protein`: about 20% of the proteins (88) are held out, grouped so that near duplicates (above 95% sequence identity) stay on the same side. Mild split.
- `cold_family`: about 20% of the proteins (88) are held out as whole kinase families (clusters at 45% identity). No held-out protein has a training protein above 0.444 identity. The tyrosine kinases all stay in training. Hard split.

Five arms are run on one split at a time: random-vector control, ESM C 300M, ESM C 600M, ProtT5-XL and the structure arm. Validation folds are also grouped, so early stopping never sees a protein from its own training part.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Config

In [ ]:
import os
from glob import glob

def find_folder(name):
    path = f"/content/drive/MyDrive/{name}"
    if not os.path.isdir(path):
        shared = glob(f"/content/drive/.shortcut-targets-by-id/*/{name}")
        if shared:
            path = shared[0]
    assert os.path.isdir(path), f"Folder not found: {path}"
    return path

DRIVE_ROOT = find_folder("Diplomska_cold")
V2_ROOT = find_folder("Diplomska_v2")
PDB_SET = "kinase_domain"
SPLIT = "cold_protein"
STOP_AFTER_FOLDS = None

os.environ["DTA_DATA_ROOT"] = DRIVE_ROOT
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("cold-split folder:", DRIVE_ROOT)
print("v2 folder (read only):", V2_ROOT)
print("split:", SPLIT, "| structure set:", PDB_SET)

cold-split folder: /content/drive/MyDrive/Diplomska_cold
v2 folder (read only): /content/drive/MyDrive/Diplomska_v2
split: cold_protein | structure set: kinase_domain


## 3. GPU

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv

Torch version: 2.11.0+cu130
CUDA available: True
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 4. Install packages

In [ ]:
!pip install -q torch_geometric biopython

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.6 MB/s eta 0:00:00


## 5. Go to the folder

In [ ]:
%cd {DRIVE_ROOT}
!pwd

/content/drive/MyDrive/Diplomska_cold
/content/drive/MyDrive/Diplomska_cold


## 6. Check the uploaded code and split files

In [ ]:
import argparse, json
import numpy as np
import pandas as pd

required = [
    "training_5_folds.py", "testing.py", "report_best_epochs.py", "compute_protein_embeddings.py",
    "src/__init__.py", "src/common/__init__.py", "src/structure/__init__.py", "src/sequence/__init__.py",
    "src/common/seeding.py", "src/common/paths.py", "src/common/run_tag.py", "src/common/split_sets.py",
    "src/common/splits.py", "src/common/structure_sets.py", "src/common/conditions.py",
    "src/common/checkpointing.py", "src/common/dta_dataset.py", "src/common/emetrics.py",
    "src/common/creating_train_and_test_set.py", "src/common/drug_representation.py",
    "src/structure/gnn_net.py", "src/structure/protein_representation.py",
    "src/sequence/plm_net.py", "src/sequence/protein_embeddings.py",
]
missing = [f for f in required if not os.path.exists(f)]
print("Missing code files:", missing or "none")
with open("src/sequence/protein_embeddings.py") as f:
    print("random_1024 control present:", "class RandomAdapter" in f.read())

from src.common.run_tag import add_run_args, run_tag
from src.common.seeding import SEED
parser = argparse.ArgumentParser()
add_run_args(parser)
help_text = parser.format_help()
print("--split flag present:", "--split" in help_text, "| accepts", SPLIT + ":", SPLIT in help_text)

from src.common.paths import paths_for
PATHS = paths_for("davis", PDB_SET, split=SPLIT)
assert PATHS.root == DRIVE_ROOT, f"Data root {PATHS.root} is not {DRIVE_ROOT}"
print("data root:", PATHS.root)
print("train folds file:", PATHS.train_folds, os.path.exists(PATHS.train_folds))
print("test fold file:  ", PATHS.test_fold, os.path.exists(PATHS.test_fold))
assert "splits" in PATHS.train_folds and SPLIT in PATHS.train_folds, "Fold files do not point at the split folder"

TAG_PART = {"cold_family": "cfam", "cold_protein": "cprot"}[SPLIT]
print("Example tags:", run_tag("plm", "esmc_300m", SEED, False, PDB_SET, split=SPLIT), "|",
      run_tag("structure", None, SEED, False, PDB_SET, split=SPLIT))
assert run_tag("structure", None, SEED, False, PDB_SET, split=SPLIT) == f"pdb2_{TAG_PART}"

Missing code files: none
random_1024 control present: True
--split flag present: True | accepts cold_protein: True
data root: /content/drive/MyDrive/Diplomska_cold
train folds file: /content/drive/MyDrive/Diplomska_cold/data/davis/splits/cold_protein/train_folds.txt True
test fold file:   /content/drive/MyDrive/Diplomska_cold/data/davis/splits/cold_protein/test_fold.txt True
Example tags: esmc_300m_pdb2_cprot | pdb2_cprot


## 7. Copy data from main directory
Copied: PDB files, contact maps, protein graphs, ligand graphs, the pair table and the stored embeddings of ESM C 300M, ESM C 600M and ProtT5-XL (so nothing is recomputed).

In [ ]:
import shutil

SRC_DAVIS = os.path.join(V2_ROOT, "data", "davis")
DST_DAVIS = os.path.join(DRIVE_ROOT, "data", "davis")
assert os.path.isdir(SRC_DAVIS), SRC_DAVIS

SKIP_TOP_LEVEL = {"pdb", "contact_maps", "protein_graphs", "protein_embeddings", "splits"}
KEEP_EMBEDDINGS = {"esmc_300m", "esmc_600m", "prot_t5_xl_uniref50"}

def ignore(dirpath, names):
    skip = set()
    if os.path.abspath(dirpath) == os.path.abspath(SRC_DAVIS):
        skip |= SKIP_TOP_LEVEL & set(names)
    if os.path.basename(dirpath) == "protein_embeddings_kinase_domain":
        skip |= set(names) - KEEP_EMBEDDINGS
    return skip

def copy_if_new(src, dst):
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        return dst
    return shutil.copy2(src, dst)

shutil.copytree(SRC_DAVIS, DST_DAVIS, ignore=ignore, copy_function=copy_if_new, dirs_exist_ok=True)

REFERENCE_DIR = os.path.join(DRIVE_ROOT, "reference_v2")
os.makedirs(REFERENCE_DIR, exist_ok=True)
os.makedirs(PATHS.models, exist_ok=True)
os.makedirs(PATHS.results, exist_ok=True)
for name in os.listdir(os.path.join(V2_ROOT, "results_davis")):
    if name.startswith("result_") and name.endswith(".txt"):
        copy_if_new(os.path.join(V2_ROOT, "results_davis", name), os.path.join(REFERENCE_DIR, name))
print("reference results copied:", len(os.listdir(REFERENCE_DIR)))

reference results copied: 16


## 8. Check the data and the split

In [ ]:
df = pd.read_csv(PATHS.csv)
names = df["protein"].unique()
pdb_missing = [n for n in names if not os.path.exists(PATHS.pdb_file(n))]
print(f"proteins in pair table: {len(names)} (expected 442)")
print(f"PDB files missing for {len(pdb_missing)} proteins <- expected 0")
print(f"protein graphs: {len(glob(os.path.join(PATHS.protein_graph_gml, '*_contact_map_graph.gml')))} (expected 442)")
print(f"ligand graphs: {len(glob(os.path.join(PATHS.ligand_graphs, '*.gml')))} (expected 68)")
for m in sorted(KEEP_EMBEDDINGS):
    print(f"{m} pooled vectors:", len(glob(os.path.join(PATHS.protein_embeddings, m, "*.pooled.pt"))), "(expected 428)")

def read_indices(path):
    with open(path) as f:
        return json.load(f)

folds = read_indices(PATHS.train_folds)
test_idx = [int(i) for i in read_indices(PATHS.test_fold)]
train_idx = [int(i) for fold in folds for i in fold]
print("train fold sizes:", [len(x) for x in folds], "| train rows:", len(train_idx), "| test rows:", len(test_idx))
assert max(train_idx + test_idx) < len(df)
train_prot = set(df.iloc[train_idx]["protein"]); test_prot = set(df.iloc[test_idx]["protein"])
print("train proteins:", len(train_prot), "| test proteins:", len(test_prot), "(expected 88)")
print("proteins in both train and test:", len(train_prot & test_prot), "<- expected 0")
fold_prot = [set(df.iloc[[int(i) for i in fold]]["protein"]) for fold in folds]
print("proteins shared between two train folds:", sum(len(fold_prot[i] & fold_prot[j]) for i in range(5) for j in range(i + 1, 5)), "<- expected 0")
print("drugs in test:", df.iloc[test_idx]["ligand"].nunique(), "| drugs in train:", df.iloc[train_idx]["ligand"].nunique(), "(expected 68, 68)")
assert not (train_prot & test_prot), "Test proteins leak into training"

proteins in pair table: 442 (expected 442)
PDB files missing for 0 proteins <- expected 0
protein graphs: 442 (expected 442)
ligand graphs: 68 (expected 68)
esmc_300m pooled vectors: 428 (expected 428)
esmc_600m pooled vectors: 428 (expected 428)
prot_t5_xl_uniref50 pooled vectors: 428 (expected 428)
train fold sizes: [4828, 4828, 4828, 4828, 4760] | train rows: 24072 | test rows: 5984
train proteins: 354 | test proteins: 88 (expected 88)
proteins in both train and test: 0 <- expected 0
proteins shared between two train folds: 0 <- expected 0
drugs in test: 68 | drugs in train: 68 (expected 68, 68)


## 9. Helpers

In [ ]:
import re, statistics, subprocess

QUIET_PREFIXES = ("Train epoch:", "Training on ", "Predicting for validation data",
                  "Make prediction for", "MSE improved at epoch", "No improvement since epoch")
VALIDATION_LINE = re.compile(r"^Epoch (\d+) - Validation MSE")

def run(args, stop_marker=None, every=10):
    cmd = ["python", "-u"] + args
    print("Running:", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    stopped = False
    try:
        for line in proc.stdout:
            match = VALIDATION_LINE.match(line)
            if match:
                if int(match.group(1)) % every == 0:
                    print(line, end="")
                continue
            if line.startswith(QUIET_PREFIXES):
                continue
            print(line, end="")
            if stop_marker and stop_marker in line:
                stopped = True
                print("\nStopping here as requested. Rerun this cell to continue; finished folds are skipped.")
                break
    finally:
        if proc.poll() is None:
            proc.terminate()
        proc.wait()
    if not stopped and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}, see the output above.")

ARMS = {
    "random_1024": ("plm", "PLMNet", "random_1024"),
    "esmc_300m": ("plm", "PLMNet", "esmc_300m"),
    "esmc_600m": ("plm", "PLMNet", "esmc_600m"),
    "prot_t5_xl_uniref50": ("plm", "PLMNet", "prot_t5_xl_uniref50"),
    "structure": ("structure", "GNNNet", None),
}

def flags(arm, seed=SEED):
    protein_repr, _, plm = ARMS[arm]
    out = ["--protein-repr", protein_repr, "--pdb-set", PDB_SET, "--split", SPLIT]
    if plm:
        out += ["--plm-model", plm]
    if seed != SEED:
        out += ["--seed", str(seed)]
    return out

def arm_tag(arm, seed=SEED):
    protein_repr, _, plm = ARMS[arm]
    tag = run_tag(protein_repr, plm, seed, False, PDB_SET, split=SPLIT)
    assert TAG_PART in tag, tag
    return tag

def stop_marker():
    return f"Training for fold {STOP_AFTER_FOLDS}..." if STOP_AFTER_FOLDS else None

def result_path(arm, seed=SEED, full=False):
    _, model_class, _ = ARMS[arm]
    return os.path.join(PATHS.results, f"result_{model_class}_davis_{arm_tag(arm, seed)}{'_full' if full else ''}.txt")

def model_path(arm, seed=SEED):
    _, model_class, _ = ARMS[arm]
    return os.path.join(PATHS.models, f"model_{model_class}_davis_{arm_tag(arm, seed)}_full.model")

def median_best_epoch(arm):
    path = os.path.join(PATHS.results, f"best_epochs_{arm_tag(arm)}.json")
    with open(path) as f:
        records = json.load(f)
    fold_ids = [str(i) for i in range(5)]
    absent = [i for i in fold_ids if i not in records]
    assert not absent, f"Folds {absent} are not finished yet in {path}"
    print(f"{'fold':>4} {'best_epoch':>10} {'last_epoch':>10} {'best_val_mse':>13}")
    for i in fold_ids:
        r = records[i]
        print(f"{i:>4} {r['best_epoch']:>10} {r['last_epoch']:>10} {r['best_val_mse']:>13.4f}")
    median = int(round(statistics.median(records[i]["best_epoch"] for i in fold_ids)))
    print("Median best epoch (use for the full-data model):", median)
    return median

def show(path):
    print(path)
    with open(path) as f:
        print(f.read())

def read_mean(path, metric):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        for line in f:
            if line.startswith(metric + " ") and "mean:" in line:
                return float(line.split("mean:")[1].split()[0])
    return None

def cv_stage(arm):
    print("=" * 70); print(f"{arm}: five folds on {SPLIT}, tag {arm_tag(arm)}")
    run(["training_5_folds.py"] + flags(arm), stop_marker=stop_marker())
    median_best_epoch(arm)
    if STOP_AFTER_FOLDS:
        return
    run(["testing.py"] + flags(arm))
    show(result_path(arm))

def full_stage(arm, seed=SEED):
    epochs = median_best_epoch(arm)
    print("=" * 70); print(f"{arm}: full-data model, seed {seed}, {epochs} epochs, tag {arm_tag(arm, seed)}")
    if os.path.exists(model_path(arm, seed)):
        print("Model already finished, skipping training:", model_path(arm, seed))
    else:
        run(["training_5_folds.py"] + flags(arm, seed) + ["--mode", "full", "--epochs", str(epochs)], every=50)
    if os.path.exists(result_path(arm, seed, full=True)):
        print("Result already exists, skipping test:", result_path(arm, seed, full=True))
    else:
        run(["testing.py"] + flags(arm, seed) + ["--full"])
    show(result_path(arm, seed, full=True))

# Baselines without protein information
The split's own training rows and test rows. A test protein is never in the training rows, so the best no-protein predictor is the mean of its drug. The arms must beat this to claim they learned anything about proteins.

In [ ]:
def concordance(y, p, chunk=500):
    y = np.asarray(y, dtype=np.float64); p = np.asarray(p, dtype=np.float64)
    num = 0.0; den = 0
    for i in range(0, len(y), chunk):
        dy = y[i:i + chunk, None] - y[None, :]
        dp = p[i:i + chunk, None] - p[None, :]
        pairs = dy > 0
        num += float((pairs & (dp > 0)).sum()) + 0.5 * float((pairs & (dp == 0)).sum())
        den += int(pairs.sum())
    return num / den

train = df.iloc[train_idx]; test = df.iloc[test_idx]
y_test = test["label"].to_numpy()
mu = train["label"].mean()
drug_mean = train.groupby("ligand")["label"].mean()
prot_mean = train.groupby("protein")["label"].mean()

predictions = {
    "global mean": np.full(len(test), mu),
    "drug mean": test["ligand"].map(drug_mean).fillna(mu).to_numpy(),
    "drug mean + protein mean (protein unseen)": (test["ligand"].map(drug_mean).fillna(mu)
                                                  + test["protein"].map(prot_mean).fillna(mu) - mu).to_numpy(),
}
print(f"split {SPLIT}: test rows {len(test)}, test proteins with a training mean: "
      f"{int(test['protein'].isin(train['protein']).sum())} rows (expected 0)")
print(f"{'predictor':45} {'MSE':>8} {'CI':>8}")
BASELINE = {}
for name, pred in predictions.items():
    mse = float(np.mean((y_test - pred) ** 2)); ci = concordance(y_test, pred)
    BASELINE[name] = (mse, ci)
    print(f"{name:45} {mse:8.4f} {ci:8.4f}")

split cold_protein: test rows 5984, test proteins with a training mean: 0 rows (expected 0)
predictor                                          MSE       CI
global mean                                     0.7806   0.5000
drug mean                                       0.6288   0.7470
drug mean + protein mean (protein unseen)       0.6288   0.7470


# Arms

## Random-vector control: make the vectors

In [ ]:
run(["compute_protein_embeddings.py", "--model", "random_1024", "--pdb-set", PDB_SET])
cache = os.path.join(PATHS.protein_embeddings, "random_1024")
print("pooled vectors:", len(glob(os.path.join(cache, "*.pooled.pt"))), "(expected 428)")

Running: python -u compute_protein_embeddings.py --model random_1024 --pdb-set kinase_domain
Structures: kinase_domain (/content/drive/MyDrive/Diplomska_cold/data/davis/pdb_kinase_domain)
442 proteins, 428 unique sequences, 0 to embed with random_1024
random_1024 pooled vectors over 428 unique sequences: mean -0.0005, std 0.9996, max |x| 4.9324, per-dimension std min 0.8854 max 1.1164
pooled vectors: 428 (expected 428)


## Random-vector control: five folds

In [ ]:
cv_stage("random_1024")

random_1024: five folds on cold_protein, tag random_1024_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --split cold_protein --plm-model random_1024
Dataset:  davis
Mode:  cv
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  random_1024
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  2000
Early stop patience:  100
Data root:  /content/drive/MyDrive/Diplomska_cold
Using device: cuda
Run tag:  random_1024_pdb2_cprot
Loaded random_1024 embeddings for 442 proteins, dim 1024
Training for fold 0...
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:114: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  valid_loader = DataLoader(
Epoch 10 - Validation MSE: 1.408483624458313, Best MSE: 1.04980349

## Random-vector control: full data

In [ ]:
full_stage("random_1024")

fold best_epoch last_epoch  best_val_mse
   0         92        192        0.8114
   1        150        250        0.5100
   2        138        238        0.7907
   3         82        182        0.6807
   4        107        207        0.5371
Median best epoch (use for the full-data model): 107
random_1024: full-data model, seed 42, 107 epochs, tag random_1024_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --split cold_protein --plm-model random_1024 --mode full --epochs 107
Dataset:  davis
Mode:  full
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  random_1024
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  107
Data root:  /content/drive/MyDrive/Diplomska_cold
Using device: cuda
Run tag:  random_1024_pdb2_cprot
Loaded random_1024 embeddings for 442 proteins, dim 1024
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecated, use

## ESM C 300M: five folds

In [ ]:
cv_stage("esmc_300m")

esmc_300m: five folds on cold_protein, tag esmc_300m_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --split cold_protein --plm-model esmc_300m
Dataset:  davis
Mode:  cv
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  esmc_300m
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  2000
Early stop patience:  100
Data root:  /content/drive/MyDrive/Diplomska_cold
Using device: cuda
Run tag:  esmc_300m_pdb2_cprot
Loaded esmc_300m embeddings for 442 proteins, dim 960
Training for fold 0...
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:114: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  valid_loader = DataLoader(
Epoch 10 - Validation MSE: 0.7645279169082642, Best MSE: 0.7834814786911011
Epo

## ESM C 300M: full data

In [ ]:
full_stage("esmc_300m")

fold best_epoch last_epoch  best_val_mse
   0        372        472        0.3291
   1        600        700        0.2547
   2        365        465        0.3703
   3        386        486        0.3186
   4        215        315        0.3378
Median best epoch (use for the full-data model): 372
esmc_300m: full-data model, seed 42, 372 epochs, tag esmc_300m_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --split cold_protein --plm-model esmc_300m --mode full --epochs 372
Dataset:  davis
Mode:  full
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  esmc_300m
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  372
Data root:  /content/drive/MyDrive/Diplomska_cold
Using device: cuda
Run tag:  esmc_300m_pdb2_cprot
Loaded esmc_300m embeddings for 442 proteins, dim 960
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecated, use 'loader.Data

## ESM C 600M: five folds

In [ ]:
cv_stage("esmc_600m")

esmc_600m: five folds on cold_protein, tag esmc_600m_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --split cold_protein --plm-model esmc_600m
Dataset:  davis
Mode:  cv
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  esmc_600m
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  2000
Early stop patience:  100
Data root:  /content/drive/MyDrive/Diplomska_cold
Using device: cuda
Run tag:  esmc_600m_pdb2_cprot
Loaded esmc_600m embeddings for 442 proteins, dim 1152
Resuming fold 0 at epoch 541 (best MSE 0.32434582710266113 at epoch 449)
Training for fold 0...
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:114: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  valid_loader = DataLoader(
Early

## ESM C 600M: full data

In [ ]:
full_stage("esmc_600m")

fold best_epoch last_epoch  best_val_mse
   0        449        549        0.3243
   1        379        479        0.2470
   2        572        672        0.3582
   3        376        476        0.3151
   4        356        456        0.3372
Median best epoch (use for the full-data model): 379
esmc_600m: full-data model, seed 42, 379 epochs, tag esmc_600m_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --split cold_protein --plm-model esmc_600m --mode full --epochs 379
Dataset:  davis
Mode:  full
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  esmc_600m
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  379
Data root:  /content/drive/MyDrive/Diplomska_cold
Using device: cuda
Run tag:  esmc_600m_pdb2_cprot
Loaded esmc_600m embeddings for 442 proteins, dim 1152
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecated, use 'loader.Dat

## ProtT5-XL: five folds

In [ ]:
cv_stage("prot_t5_xl_uniref50")

prot_t5_xl_uniref50: five folds on cold_protein, tag prot_t5_xl_uniref50_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --split cold_protein --plm-model prot_t5_xl_uniref50
Dataset:  davis
Mode:  cv
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  prot_t5_xl_uniref50
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  2000
Early stop patience:  100
Data root:  /content/drive/MyDrive/Diplomska_cold
Using device: cuda
Run tag:  prot_t5_xl_uniref50_pdb2_cprot
Loaded prot_t5_xl_uniref50 embeddings for 442 proteins, dim 1024
Training for fold 0...
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:114: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  valid_loader = DataLoader(
Epoch 10 - Validat

## ProtT5-XL: full data

In [ ]:
full_stage("prot_t5_xl_uniref50")

fold best_epoch last_epoch  best_val_mse
   0        486        586        0.3306
   1        407        507        0.2671
   2        419        519        0.3680
   3        375        475        0.3113
   4        150        250        0.3773
Median best epoch (use for the full-data model): 407
prot_t5_xl_uniref50: full-data model, seed 42, 407 epochs, tag prot_t5_xl_uniref50_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --split cold_protein --plm-model prot_t5_xl_uniref50 --mode full --epochs 407
Dataset:  davis
Mode:  full
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  prot_t5_xl_uniref50
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  407
Data root:  /content/drive/MyDrive/Diplomska_cold
Using device: cuda
Run tag:  prot_t5_xl_uniref50_pdb2_cprot
Loaded prot_t5_xl_uniref50 embeddings for 442 proteins, dim 1024
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:42: U

## Structure: five folds

In [ ]:
cv_stage("structure")

structure: five folds on cold_protein, tag pdb2_cprot
Running: python -u training_5_folds.py --protein-repr structure --pdb-set kinase_domain --split cold_protein
Dataset:  davis
Mode:  cv
Protein representation:  structure
Structure set:  kinase_domain
Seed:  42
Learning rate:  0.001
Epochs:  2000
Early stop patience:  100
Data root:  /content/drive/MyDrive/Diplomska_cold
Using device: cuda
Run tag:  pdb2_cprot
GNNNet Loaded
Resuming fold 0 at epoch 761 (best MSE 0.5246933102607727 at epoch 669)
Training for fold 0...
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:114: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  valid_loader = DataLoader(
Epoch 770 - Validation MSE: 0.6004820466041565, Best MSE: 0.5241873264312744
Epoch 780 - Validation MSE: 0.5888622403144836, Best MSE: 0.5241873

## Structure: full data

In [ ]:
full_stage("structure")

fold best_epoch last_epoch  best_val_mse
   0        847        947        0.5211
   1        293        393        0.4302
   2        104        204        0.7073
   3        337        437        0.4816
   4        250        350        0.4261
Median best epoch (use for the full-data model): 293
structure: full-data model, seed 42, 293 epochs, tag pdb2_cprot
Running: python -u training_5_folds.py --protein-repr structure --pdb-set kinase_domain --split cold_protein --mode full --epochs 293
Dataset:  davis
Mode:  full
Protein representation:  structure
Structure set:  kinase_domain
Seed:  42
Learning rate:  0.001
Epochs:  293
Data root:  /content/drive/MyDrive/Diplomska_cold
Using device: cuda
Run tag:  pdb2_cprot
GNNNet Loaded
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
Saved full-data model after 293 epochs to /content/drive/MyDrive/Diplomska_cold/models_davis/model_G

# Seeds 43 and 44 of the full-data models

In [ ]:
SEEDS = [43, 44]
SEED_ARMS = ["esmc_600m", "esmc_300m", "prot_t5_xl_uniref50", "random_1024", "structure"]   # add "structure" for the slow arm
for arm in SEED_ARMS:
    for seed in SEEDS:
        full_stage(arm, seed)

fold best_epoch last_epoch  best_val_mse
   0        449        549        0.3243
   1        379        479        0.2470
   2        572        672        0.3582
   3        376        476        0.3151
   4        356        456        0.3372
Median best epoch (use for the full-data model): 379
esmc_600m: full-data model, seed 43, 379 epochs, tag esmc_600m_pdb2_cprot_s43
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --split cold_protein --plm-model esmc_600m --seed 43 --mode full --epochs 379
Dataset:  davis
Mode:  full
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  esmc_600m
Standardized embeddings:  False
Seed:  43
Learning rate:  0.001
Epochs:  379
Data root:  /content/drive/MyDrive/Diplomska_cold
Using device: cuda
Run tag:  esmc_600m_pdb2_cprot_s43
Loaded esmc_600m embeddings for 442 proteins, dim 1152
/content/drive/MyDrive/Diplomska_cold/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecate

# Summary

In [ ]:
RANDOM_SPLIT_FULL = {"random_1024": (0.2096, 0.8915), "esmc_300m": (0.1654, 0.9100), "esmc_600m": (0.1623, 0.9110),
                     "prot_t5_xl_uniref50": (0.1699, 0.9099), "structure": (0.1691, 0.9115)}
fmt = lambda v: f"{v:8.4f}" if v is not None else f"{'-':>8}"

print(f"split: {SPLIT}  ({len(test_prot)} held-out proteins, {len(test)} test rows)")
print(f"{'arm':26} {'5-fold MSE':>10} {'5-fold CI':>10} {'full MSE':>10} {'full CI':>10}   {'3-seed full MSE (std)':>22}")
for name, (m, c) in BASELINE.items():
    print(f"{name:26} {'':>10} {'':>10} {fmt(m):>10} {fmt(c):>10}   (no network)")
for arm in ARMS:
    m5 = read_mean(result_path(arm), "mse"); c5 = read_mean(result_path(arm), "ci")
    mf = read_mean(result_path(arm, full=True), "mse"); cf = read_mean(result_path(arm, full=True), "ci")
    vals = [read_mean(result_path(arm, s, full=True), "mse") for s in [SEED] + SEEDS]
    vals = [v for v in vals if v is not None]
    seeds_txt = f"{statistics.mean(vals):.4f} ({statistics.stdev(vals):.4f}, n={len(vals)})" if len(vals) >= 2 else "-"
    print(f"{arm:26} {fmt(m5):>10} {fmt(c5):>10} {fmt(mf):>10} {fmt(cf):>10}   {seeds_txt:>22}")
print()
print("Random split, full data, 3 seeds (different test set, context only):")
for arm, (m, c) in RANDOM_SPLIT_FULL.items():
    print(f"  {arm:24} MSE {m:.4f}  CI {c:.4f}")

split: cold_protein  (88 held-out proteins, 5984 test rows)
arm                        5-fold MSE  5-fold CI   full MSE    full CI    3-seed full MSE (std)
global mean                                          0.7806     0.5000   (no network)
drug mean                                            0.6288     0.7470   (no network)
drug mean + protein mean (protein unseen)                           0.6288     0.7470   (no network)
random_1024                    0.6628     0.7233     0.7372     0.7250     0.8038 (0.1738, n=3)
esmc_300m                      0.4015     0.8194     0.3938     0.8334     0.4485 (0.0498, n=3)
esmc_600m                      0.3651     0.8352     0.4278     0.8245     0.4661 (0.0385, n=3)
prot_t5_xl_uniref50            0.4219     0.8233     0.5195     0.8193     0.5274 (0.0254, n=3)
structure                      0.6348     0.7519     0.7034     0.7248     0.7110 (0.0418, n=3)

Random split, full data, 3 seeds (different test set, context only):
  random_1024        